## Convolutional Neural Network

A **Convolutional Neural Network (CNN or ConvNet)** is a specialized class of deep neural network designed primarily for processing grid-like structured data—most notably **images, videos, and spatial data**.

While a standard Multi-Layer Perceptron (MLP) treats every input pixel or feature independently, CNNs are engineered to automatically and adaptively learn spatial hierarchies of features, from low-level edges and textures up to complex object shapes and faces.

---

### Why Standard MLPs Fail on Images

If you tried to feed a high-resolution color image (e.g., $224 \times 224$ pixels) into a standard MLP:

1. **Parameter Explosion:** A single hidden layer connected to every pixel would require millions of weights just for the first layer, leading to massive memory overhead and severe overfitting.
2. **Loss of Spatial Structure:** Flattening a 2D image into a 1D vector destroys the crucial spatial relationships between adjacent pixels (e.g., knowing that a pixel is right next to another pixel).

---

### Core Building Blocks of a CNN

A typical CNN architecture combines three main types of layers:

1. **Convolutional Layers (The Feature Extractors):**
* Instead of connecting every neuron, CNNs use small filters (or **kernels**, e.g., $3 \times 3$ matrices) that slide (convolve) across the image.
* By computing dot products between the filter and local patches of the image, these layers detect local patterns like edges, gradients, corners, and textures.
* **Key Advantage (Weight Sharing):** The same filter is reused across the entire image, drastically reducing the total number of parameters.


2. **Pooling Layers (Downsampling):**
* Layers like **Max Pooling** or Average Pooling reduce the spatial dimensions (height and width) of the feature maps.
* This cuts down computational complexity, controls overfitting, and provides **translation invariance** (recognizing an object whether it's in the center or corner of the frame).


3. **Fully Connected (Dense) Layers:**
* Positioned at the very end of the network (which is often an **MLP**!), these layers take the high-level abstract features extracted by the convolutional and pooling layers and output the final classification or regression prediction.

To understand how a Convolutional Neural Network (CNN) extracts patterns from an image, we need to look under the hood at its three core mechanics: the **2D Convolution Operation**, **Stride**, and **Padding**.

![Mapping an input matrix to an output feature map using a kernel. Source: Made with Google AI](https://encrypted-tbn0.gstatic.com/licensed-image?q=tbn:ANd9GcRGe4RhVwBpNkn3bP5iL0hZIhco9UEhVEdD_hotxI6zlEHNwJ8biY40fMOvhLVYTwYiAosxGkSbNcauKRM)

---

### 1. The 2D Convolution Operation

Instead of processing an entire image at once, a CNN uses a small matrix called a **filter** (or **kernel**)—typically sized $3 \times 3$ or $5 \times 5$.

This filter slides across the 2D input image. At every position, it performs an **element-wise multiplication** between its own weights and the underlying pixel values, then sums them all up into a single scalar value. This resulting matrix of values is called a **feature map**.

---

### 2. Stride ($S$)

**Stride** defines how many pixels the filter shifts horizontally or vertically at each step:

* **Stride = 1:** The filter moves 1 pixel to the right (or down), resulting in high overlap between adjacent receptive fields and preserving fine details.
* **Stride = 2 (or more):** The filter skips pixels, moving faster across the image. This shrinks the output feature map size and reduces computational cost.

---

### 3. Padding ($P$)

When a filter slides over an image, **pixels on the outer edges are touched fewer times** than pixels in the center. Furthermore, applying convolutions repeatedly causes the image dimensions to shrink rapidly layer by layer.

**Padding** solves this by adding a border of pixels (usually zeros) around the edges of the input matrix:

* **Valid Padding ($P = 0$):** No padding added. The output shrinks at every layer.
* **Same Padding ($P > 0$):** Adds enough border pixels so that the output feature map has the exact same spatial dimensions as the input.

---

### Step-by-Step Numerical Example

Let's walk through a concrete example with numbers:

* **Input Image ($W$):** $4 \times 4$ matrix
* **Kernel/Filter Size ($K$):** $3 \times 3$ matrix
* **Stride ($S$):** $1$
* **Padding ($P$):** $0$

#### The Input Matrix ($4 \times 4$)

$$\begin{bmatrix} 1 & 0 & 2 & 3 \\  4 & 6 & 1 & 0 \\  2 & 3 & 5 & 1 \\  1 & 1 & 0 & 2 \end{bmatrix}$$

#### The Kernel ($3 \times 3$)

$$\begin{bmatrix} 1 & 0 & -1 \\  1 & 1 & 0 \\  0 & -1 & 1 \end{bmatrix}$$

#### Calculation Steps:

1. **Top-Left Corner ($3 \times 3$ region):**
We place the kernel over the top-left $3 \times 3$ patch of the input:

$$\begin{bmatrix} 1 & 0 & 2 \\ 4 & 6 & 1 \\ 2 & 3 & 5 \end{bmatrix}$$



Multiply element-by-element with the kernel and sum them up:

$$(1\times1) + (0\times0) + (2\{-1\}) + (4\times1) + (6\times1) + (1\times0) + (2\times0) + (3\{-1\}) + (5\times1)$$


$$= 1 + 0 - 2 + 4 + 6 + 0 + 0 - 3 + 5 = 11$$



This gives the **first top-left value** ($11$) of our output feature map.
2. **Moving Across:**
The filter shifts right by **Stride = 1** to cover the next $3 \times 3$ patch, computes the dot product, and places the result in the top-right of the output.
3. **Moving Down:**
Once it hits the right edge, it resets to the left and shifts down by **Stride = 1**.

Because a $3 \times 3$ filter can only fit into a $4 \times 4$ grid twice horizontally and twice vertically, our output feature map will be a **$2 \times 2$ matrix**.

---

### The Universal Output Dimension Formula

To calculate the exact size of your output feature map without manual grid counting, use this standard formula:

$$\text{Output Size} = \left\lfloor \frac{W - K + 2P}{S} \right\rfloor + 1$$

* Where $W$ = Input width/height, $K$ = Kernel size, $P$ = Padding, $S$ = Stride.
* Plugging in our example above:

$$\left\lfloor \frac{4 - 3 + (2 \times 0)}{1} \right\rfloor + 1 = 1 + 1 = 2$$



(yielding a $2 \times 2$ output).

### Pooling Layers in CNNs: Explanation & Purpose

After a convolutional layer extracts feature maps (detailing edges, textures, and patterns), a **Pooling Layer** is typically applied to downsample those feature maps.

Just like convolutions, a pooling layer slides a window (e.g., $2 \times 2$) across the feature map with a stride, but instead of taking a weighted dot product, it performs a simple aggregation function:

1. **Max Pooling (Most Common):**
* Extracts the **maximum value** from each local pooling window.
* *Why it's used:* It acts like a noise suppressor, retaining only the strongest, most prominent features (such as a distinct edge or high-contrast corner) while discarding weaker background details.


2. **Average Pooling:**
* Calculates the **average value** of all elements inside the pooling window.
* *Why it's used:* It smooths out the feature map, preserving broader regional context rather than peak activations.



#### Why Do We Need Pooling?

* **Dimensionality Reduction:** By cutting the height and width dimensions in half (e.g., from $28 \times 28$ down to $14 \times 14$), it drastically reduces the computational cost and memory required for subsequent layers.
* **Translation Invariance:** If a feature (like a cat's ear) shifts slightly to the left or right in the image, its maximum value within the local pooling window will often remain unchanged, making the network robust to spatial shifts.
* **Expanding Receptive Field:** Downsampling allows deeper layers to look at a larger global context of the image.

---

### Python Implementation (PyTorch)

Here is a clean PyTorch implementation showing a small CNN block that combines a **2D Convolutional layer**, a **ReLU activation**, and a **Max Pooling layer**. We will trace the tensor shapes step-by-step so you can see exactly how pooling shrinks the dimensions.